In [ ]:
## Step 3: Distance Transform analysis (Jupyter Notebook)

import pandas as pd
import numpy as np
import cv2
import tifffile
import re
from pathlib import Path
from scipy.stats import linregress
from read_roi import read_roi_zip
from skimage.draw import polygon
import matplotlib.pyplot as plt  # Added to generate plots

# --- 1. HELPER FUNCTIONS ---

def sanitize_filename(name):
    # Removes characters that are not alphanumeric, dashes, or underscores
    return re.sub(r'[^\w\-]', '_', name)

def rois_to_labels(rois, shape):
    labels = np.zeros(shape, dtype=np.int32)
    label_id = 1
    for roi_name, roi_data in rois.items():
        if roi_data['type'] in ['polygon', 'freehand', 'traced']:
            x, y = np.array(roi_data['x']), np.array(roi_data['y'])
            rr, cc = polygon(y, x, shape)
            labels[rr, cc] = label_id
            label_id += 1
        elif roi_data['type'] == 'rectangle':
            top, left = roi_data['top'], roi_data['left']
            width, height = roi_data['width'], roi_data['height']
            labels[top:top+height, left:left+width] = label_id
            label_id += 1
    return labels

# --- 2. PARAMETERS ---

experiment_folders = [
    {
        'time_label': '4h',
        'images_dir': r"enhanced_resolution_iamge", 
        'rois_dir':   r"5x_rois"    
    }
]

valid_suffix = ['.tif', '.tiff']
output_folder = Path(r"")
output_folder.mkdir(parents=True, exist_ok=True) # Ensures the main folder exists

# --- NEW: PARAMETERS FOR SVG FIGURES ---
plots_folder = output_folder / "SVG_Figures"
plots_folder.mkdir(parents=True, exist_ok=True)

# PLACE THE EXACT IDs OF THE CELLS YOU WANT TO PLOT HERE
CELLS_FOR_FIGURE = [
    "110_1_c1_MSSR", # Example: Replace with an actual ID from your CSV
    "110_2_c1_MSSR"  # Example 2
]

PIXEL_SIZE_NM = 23.4
DISPERSED_THRESHOLD = 0.10 
FIXED_THRESHOLD = 10 

# --- 3. PROCESSING ---

summary_results = []
raw_data_list = []  

print("Starting analysis (Intensity Normalized, Generating SVG for selected cells)...")

for folder_data in experiment_folders:
    condition = folder_data['time_label']
    img_path = Path(folder_data['images_dir'])
    roi_path = Path(folder_data['rois_dir'])

    if not img_path.exists() or not roi_path.exists():
        print(f"Skipping {condition}: Path not found.")
        continue
        
    for file_path in img_path.iterdir():
        if file_path.suffix not in valid_suffix: continue
            
        base_name = file_path.stem
        parts = base_name.split('_')
        if len(parts) < 2: continue
            
        zip_path = roi_path / f"{parts[0]}_{parts[1]}_roi_5x.zip"
        if not zip_path.exists(): continue
            
        rois = read_roi_zip(zip_path)
        image = tifffile.imread(file_path)
        labels = rois_to_labels(rois, image.shape)

        for label_id in range(1, np.amax(labels) + 1):
            mask = (labels == label_id).astype(np.uint8)
            total_pixels = np.count_nonzero(mask)
            if total_pixels == 0: continue
            
            # --- DISTANCE TRANSFORM ---
            cell_DT = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
            cell_DT = cv2.normalize(cell_DT, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
            
            mask_signal = (mask > 0) & (image > FIXED_THRESHOLD) & (cell_DT > 1)
            intensity_raw = image[mask_signal].flatten()
            dt_vals = cell_DT[mask_signal].flatten()

            slope, intercept, r_squared, origin, relative_diff = 0, 0, 0, "Unknown", 0
            
            # Unique cell ID
            unique_id = sanitize_filename(f"{condition}_{base_name}_Cell_{label_id}")
            
            if len(dt_vals) >= 10: 
                # --- INTENSITY NORMALIZATION (Min-Max 0-1) ---
                int_min = np.min(intensity_raw)
                int_max = np.max(intensity_raw)
                
                if int_max > int_min:
                    intensity_norm = (intensity_raw - int_min) / (int_max - int_min)
                else:
                    intensity_norm = np.zeros_like(intensity_raw, dtype=float)

                # --- REGRESSION ON NORMALIZED DATA ---
                df_temp = pd.DataFrame({'dt': dt_vals, 'int_norm': intensity_norm})
                grouped = df_temp.groupby('dt')['int_norm'].mean()
                unique_x, mean_y_norm = grouped.index.values, grouped.values
                
                slope, intercept, r_val, _, _ = linregress(unique_x, mean_y_norm)
                r_squared = r_val**2
                
                # Relative difference adjusted for normalized data
                absolute_difference = slope * (unique_x[-1] - unique_x[0])
                relative_diff = absolute_difference / (np.mean(intensity_norm) + 1e-6)
                
                # Classification logic: Negative slope indicates membrane proximity
                if abs(relative_diff) < DISPERSED_THRESHOLD: origin = "Dispersed"
                elif relative_diff < 0: origin = "Membrane"
                else: origin = "Cytoplasm"
                
                # --- NEW: SVG PLOT GENERATION FOR SPECIFIC CELLS ---
                if unique_id in CELLS_FOR_FIGURE:
                    plt.figure(figsize=(6, 4))
                    
                    # Plot data points and regression line
                    plt.scatter(unique_x, mean_y_norm, color='#000000', alpha=0.6, s=15, label='Mean Intensity')
                    y_pred = slope * unique_x + intercept
                    
                    # Using Okabe-Ito Vermillion (#D55E00) for the fit line
                    plt.plot(unique_x, y_pred, color='#D55E00', linewidth=2, 
                             label=f'Linear Fit\nm={slope:.4f}, $R^2$={r_squared:.2f}')
                    
                    # Figure aesthetics
                    plt.title(f'Spatial Profile - {unique_id}\nClassification: {origin}', fontsize=10)
                    plt.xlabel('Relative Distance (0 = Membrane, 255 = Center)', fontsize=10)
                    plt.ylabel('Normalized Intensity (0 to 1)', fontsize=10)
                    plt.ylim(-0.05, 1.05) 
                    
                    plt.legend(frameon=False, fontsize=9)
                    plt.grid(False) 
                    plt.tight_layout()
                    
                    # Save in SVG format
                    plt.savefig(plots_folder / f"{unique_id}_plot.svg", format='svg', bbox_inches='tight')
                    plt.close() # Close the figure to prevent memory saturation

            # --- DATA COLLECTION ---
            summary_results.append({
                'Condition': condition, 
                'Cell_ID': unique_id, 
                'Origin': origin, 
                'Slope_Norm': slope,
                'R_squared': r_squared,
                'Relative_Diff': relative_diff
            })
            
            if len(dt_vals) >= 10:
                raw_data_list.append(pd.DataFrame({
                    'ID': unique_id, 
                    'DT': dt_vals, 
                    'Raw_Int': intensity_raw,
                    'Norm_Int': intensity_norm
                }))

# --- 4. EXPORT ---
if summary_results:
    print("Exporting data to Excel and CSV...")
    pd.DataFrame(summary_results).to_excel(output_folder / 'Summary.xlsx', index=False)
    pd.concat(raw_data_list).to_csv(output_folder / 'Raw_Data.csv', index=False)
    print(f"Process finished successfully. Files saved in: {output_folder}")
else:
    print("No valid cells found to analyze.")
